<a href="https://colab.research.google.com/github/sergioGarcia91/SeismicUP/blob/main/02_Replica_Londo%C3%B1o_etal_2019.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

En este Notebook se seleccionarán los datos de los catálogos sísmicos y se filtrarán usando los mismos límites espaciales por Londoño et al. (2019). El objetivo es replicar sus resultados y comprobar si, al aplicar nuestras ecuaciones y procedimientos sobre el conjunto de datos, se obtiene un comportamiento consistente.

---
Londoño, J. M., Quintero, S., Vallejo, K., Muñoz, F., & Romero, J. (2019). Seismicity of Valle Medio del Magdalena basin, Colombia. Journal of South American Earth Sciences, 92, 565–585. https://doi.org/10.1016/j.jsames.2019.04.003

L0NDONO, J. M. (2019). DISTRIBUICIÓN ESPACIAL 3D DEL VALOR-B EN EL SECTOR DEL VALLE MEDIO DEL MAGDALENA, COLOMBIA. Geosciences = Geociências, 38(1), 171–183. https://doi.org/10.5016/geociencias.v38i1.12386



# Inicio

In [ ]:
!git clone https://github.com/sergioGarcia91/SeismicUP.git

In [ ]:
# para incluir la libreria clonada
import sys
sys.path.append("/content/SeismicUP")

import seismicup as sup

In [ ]:
!pip -q install python-calamine

In [ ]:
!pip3 install contextily

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
import seaborn as sns
import os
import re
import contextily as cx #para el basemap en geopandas
import xyzservices.providers as xyz #para escoger el basemap
import seismicup as sup
import urllib.request
import matplotlib.font_manager as fm

from matplotlib.colors import LogNorm # para la escala logaritmica de los colores
from matplotlib.ticker import LogFormatter, LogLocator # escala log
from scipy import stats # regresion lineal

# Conectar al Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Cambiar Fuente

In [ ]:
sup.plots.get_TimesNewRoman_font()

# Paths

In [ ]:
path_save_figures = '/content/drive/MyDrive/Contratos/20250801_UIS_CO2/Notebooks/Figuras_02'

In [ ]:
path_datasets = '/content/SeismicUP/Datasets_'

os.listdir(path_datasets)

# Cargar catalogos

In [ ]:
lista_catalogos = ['Cat_sis_SGC_jun1993_feb2018',
                   'Cat_sis_SGC_mar2018_2024',
                   'Cat_sis_SGC_TECTO_feb2014_2024',
                   'Cat_sis_SGC_LBG_jun1993_2021']

for i in lista_catalogos:
  cantidad_archivos = len(os.listdir(os.path.join(path_datasets, i)))
  print(f'{i} = ', cantidad_archivos)

In [ ]:
df_general = pd.DataFrame(columns=['Fecha-Hora UTC',
                                   'Latitud', # grados
                                   'Longitud', # grados
                                   'Profundidad [km]',
                                   'Magnitud',
                                   'Tipo Magnitud',
                                   'Error Latitud [km]',
                                   'Error Longitud [km]',
                                   'Error Profundidad [km]',
                                   'Numero de Fases',
                                   'RMS [seg]',
                                   'Gap', # grados
                                   'Departamento',
                                   'Municipio',
                                   ])

df_general.head()

## Catalogo SGC 1

In [ ]:
path_catalogo = os.path.join(path_datasets, 'Cat_sis_SGC_jun1993_feb2018')
os.listdir(path_catalogo)

df_SGC_1 = df_general.copy()

for i in os.listdir(path_catalogo)[:]:
  if i.endswith('.xlsx'):
    print('-----'*3)
    print('File: ', i)
    print('-----'*3)
    df_temp_1 = sup.io.crear_catalogo_SGC_1(path_catalogo=path_catalogo,
                                            file_excel=i)

    df_SGC_1 = pd.concat([df_SGC_1, df_temp_1], ignore_index=True)
    df_SGC_1.reset_index(drop=True, inplace=True)

    del df_temp_1


In [ ]:
df_SGC_1['Catalogo'] = 'SGC 1'
df_SGC_1['Gap'] = df_SGC_1['Gap'].astype(float)
df_SGC_1.info()

In [ ]:
df_SGC_1.head()

## Catalogo SGC 2

In [ ]:
path_catalogo = os.path.join(path_datasets, 'Cat_sis_SGC_mar2018_2024')
os.listdir(path_catalogo)

df_SGC_2 = df_general.copy()

for i in os.listdir(path_catalogo)[:]:
  if i.endswith('.xlsx'):
    print('-----'*3)
    print('File: ', i)
    print('-----'*3)
    df_temp_1 = sup.io.crear_catalogo_SGC_2(path_catalogo=path_catalogo,
                                            file_excel=i)

    df_SGC_2 = pd.concat([df_SGC_2, df_temp_1], ignore_index=True)
    df_SGC_2.reset_index(drop=True, inplace=True)

    del df_temp_1


In [ ]:
df_SGC_2['Catalogo'] = 'SGC 2'
df_SGC_2['Gap'] = df_SGC_2['Gap'].astype(float)
df_SGC_2.info()

In [ ]:
df_SGC_2.head()

## Catalogo TECTO

In [ ]:
path_catalogo = os.path.join(path_datasets, 'Cat_sis_SGC_TECTO_feb2014_2024')
os.listdir(path_catalogo)

df_SGC_TECTO = df_general.copy()

for i in os.listdir(path_catalogo)[:]:
  if i.endswith('.xlsx') and (i != 'excel_estaciones.xlsx'):
    print('-----'*3)
    print('File: ', i)
    print('-----'*3)
    df_temp_1 = sup.io.crear_catalogo_SGC_TECTO(path_catalogo=path_catalogo,
                                                file_excel=i)

    print(df_temp_1.shape)
    df_SGC_TECTO = pd.concat([df_SGC_TECTO, df_temp_1], ignore_index=True)
    df_SGC_TECTO.reset_index(drop=True, inplace=True)

    del df_temp_1


In [ ]:
df_SGC_TECTO['Catalogo'] = 'TECTO'
df_SGC_TECTO['Profundidad [km]'] = df_SGC_TECTO['Profundidad [km]'].astype(float)
df_SGC_TECTO['Gap'] = df_SGC_TECTO['Gap'].astype(float)
df_SGC_TECTO.info()

In [ ]:
df_SGC_TECTO.head()

## Catalogo LBG

In [ ]:
path_catalogo = os.path.join(path_datasets, 'Cat_sis_SGC_LBG_jun1993_2021')
os.listdir(path_catalogo)

df_SGC_LBG = df_general.copy()

for i in os.listdir(path_catalogo)[:]:
  if i.endswith('.xlsx'):
    print('-----'*3)
    print('File: ', i)
    print('-----'*3)
    df_temp_1 = sup.io.crear_catalogo_SGC_LBG(path_catalogo=path_catalogo,
                                              file_excel=i)

    print(df_temp_1.shape)
    df_SGC_LBG = pd.concat([df_SGC_LBG, df_temp_1], ignore_index=True)
    df_SGC_LBG.reset_index(drop=True, inplace=True)

    del df_temp_1


In [ ]:
df_SGC_LBG['Catalogo'] = 'LBG'
df_SGC_LBG.info()

In [ ]:
df_SGC_LBG.head()

# Unir catalogos

In [ ]:
df_unido = pd.concat([df_SGC_1, df_SGC_2, df_SGC_TECTO, df_SGC_LBG], ignore_index=True)
df_unido.reset_index(drop=True, inplace=True)

df_unido['Tiempo'] = pd.to_datetime(df_unido['Fecha-Hora UTC'])

df_unido.head()

In [ ]:
df_unido['Catalogo'].unique()

In [ ]:
df_unido.info()

In [ ]:
df_unido.describe().round(2)

## Limites Londoño et al. (2019)

Londoño et al. (2019) reportan un total de 7394 eventos. Sin embargo, al aplicar los criterios de selección para la red local del VMM en el periodo 2014–2017, en este análisis se obtuvieron 7073 eventos.

Usan magnitud local ML.

Para la Red Sismológica Nacional, Londoño et al. (2019) seleccionaron 26463 eventos correspondientes al periodo 1993–2014.

Al clasificar por profundidad usando un umbral de 50 km, identifican 2059 eventos con profundidad < 50 km (corteza) y 5335 eventos asociados a la zona de subducción (50–160 km).

In [ ]:
years_ = [2014, 2017] #
longitud_ = [-74.5, -73] # N
latitud_ = [6.4, 9.2] # W
prof_max = 160 # km
rms_ = 0.4 # s
erros_loc_ = 10 # km
gap_ = 180
ml_ = [0.1, 5.7]

In [ ]:
df_filtrado = df_unido.copy()

df_filtrado = df_filtrado[(df_filtrado['Tiempo'].dt.year >= years_[0]) & (df_filtrado['Tiempo'].dt.year <= years_[1])]
df_filtrado = df_filtrado[(df_filtrado['Longitud'] >= longitud_[0]) & (df_filtrado['Longitud'] <= longitud_[1])]
df_filtrado = df_filtrado[(df_filtrado['Latitud'] >= latitud_[0]) & (df_filtrado['Latitud'] <= latitud_[1])]
df_filtrado = df_filtrado[df_filtrado['Profundidad [km]'] <= prof_max]
df_filtrado = df_filtrado[df_filtrado['RMS [seg]'] <= rms_]
df_filtrado = df_filtrado[(df_filtrado['Error Longitud [km]'] <= erros_loc_) & (df_filtrado['Error Latitud [km]'] <= erros_loc_ ) & (df_filtrado['Error Profundidad [km]'] <= erros_loc_)]
df_filtrado = df_filtrado[df_filtrado['Gap'] <= gap_]
df_filtrado = df_filtrado[(df_filtrado['Magnitud'] >= ml_[0]) & (df_filtrado['Magnitud'] <= ml_[1])]

df_filtrado.describe().round(2)

In [ ]:
df_filtrado['Catalogo'].unique()

Debido al rango temporal analizado, el catálogo SGC-2 no aparece incluido.

In [ ]:
df_filtrado['Catalogo'].value_counts().sort_values(ascending=False)

In [ ]:
df_filtrado['Tipo Magnitud'].value_counts().sort_values(ascending=False)

In [ ]:
df_filtrado[df_filtrado['Catalogo'] == 'SGC 1'].describe().round(2)

In [ ]:
df_filtrado[df_filtrado['Catalogo'] == 'TECTO'].describe().round(2)

In [ ]:
df_filtrado[df_filtrado['Catalogo'] == 'LBG'].describe().round(2)

Los catálogos TECTO y LBG no incluyen eventos con profundidades mayores a 50 km y, además, su número de eventos no es cercano a 2059, por lo que no es consistente asumir que correspondan al subconjunto cortical (<50 km) reportado por Londoño et al. (2019). En consecuencia, para esta replica se trabajará únicamente con el catálogo SGC-1.

In [ ]:
df_filtrado.duplicated(subset=['Fecha-Hora UTC', 'Latitud', 'Longitud'], keep='first').sum()
# 1419

# SGC-1

In [ ]:
df_filtrado_SGC1 = df_filtrado[df_filtrado['Catalogo'] == 'SGC 1'].copy()
df_filtrado_SGC1.reset_index(drop=True, inplace=True)

df_filtrado_SGC1.describe().round(2)

## Duplicados

Búsqueda y depuración de duplicados en el catálogo.

In [ ]:
df_filtrado_SGC1.duplicated(subset=['Fecha-Hora UTC', 'Latitud', 'Longitud'], keep='first').sum()
# 267 duplicados

In [ ]:
df_filtrado_SGC1['Tipo Magnitud'].unique()

Dado que Londoño et al. (2019) trabajan exclusivamente con la magnitud local (ML), en este Notebook se seleccionarán únicamente los eventos que cuenten con este tipo de magnitud.

In [ ]:
df_filtrado_SGC1 = df_filtrado_SGC1[df_filtrado_SGC1['Tipo Magnitud'] == 'Ml']
df_filtrado_SGC1.reset_index(drop=True, inplace=True)

df_filtrado_SGC1['Tipo Magnitud'].unique()

In [ ]:
df_filtrado_SGC1.duplicated(subset=['Fecha-Hora UTC', 'Latitud', 'Longitud'], keep='first').sum()
# Ya no se tienen duplicados

In [ ]:
df_filtrado_SGC1.describe().round(2)

## Calculo Mc

Londoño et al. (2019) reportaron un valor de Mc = 1.0. A continuación, se evaluarán las ecuaciones propuestas para determinar cuál de ellas reproduce un valor cercano a este estimado.

In [ ]:
N_lgr = sup.stats.obtener_N_LGR(magnitudes_array= df_filtrado_SGC1['Magnitud'],
                                magnitud_minima=0,
                                magnitud_maxima=10,
                                steps_=0.1,
                                texto=True)

N_lgr

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 4))

sup.plots.GutenbergRichter_scatter_plot(N_count = N_lgr['N'],
                                        magnitudes= N_lgr['m'],
                                        ax_h= ax,
                                        mc=1.0,
                                        magnitud_maxima=10)

sup.plots.GutenbergRichter_line_plot(b_value=1.02,
                                     a_value=5.3,
                                     ax_h=ax,
                                     magnitud_minima=1.0,
                                     magnitud_maxima=4.7)

plt.show()

In [ ]:
mc_estabilidad = sup.stats.mc_estabilidad_b(magnitudes= N_lgr['m'],
                                            valores_N = N_lgr['N'],
                                            magnitud_minima = 0,
                                            magnitud_maxima = 10,
                                            ventana_estabilidad = 10,
                                            redondeo_ = 3,
                                            min_mc_top3 = True,
                                            texto = True)


mc_estabilidad

In [ ]:
mc_max_log10 = sup.stats.mc_maximo_N_log10(magnitudes= N_lgr['m'],
                                           valores_N = N_lgr['N'],
                                           correcion_m = 0.2,
                                           magnitud_minima = 0,
                                           magnitud_maxima = 10,
                                           redondeo_N = 1)

mc_max_log10

In [ ]:
mc_max_curv = sup.stats.mc_maxima_curvatura(magnitudes_array= df_filtrado_SGC1['Magnitud'],
                                             steps_=0.1,
                                             correcion_m = 0.2,
                                             magnitud_minima = 0,
                                             magnitud_maxima = 10,
                                             texto = True)

mc_max_curv

In [ ]:
mc_menor_pendiente = sup.stats.mc_menor_pendiente(magnitudes= N_lgr['m'],
                                                  valores_N = N_lgr['N'],
                                                  magnitud_minima = 0,
                                                  magnitud_maxima = 10,
                                                  texto = True)

mc_menor_pendiente

In [ ]:
print('Valores de Mc')
print('1.00, Londoño et al. (2019)')
print(f'{mc_estabilidad['mc']:.2f}, estabilidad de b')
print(f'{mc_max_log10:.2f}, Mc maximo N Log10')
print(f'{mc_max_curv:.2f}, maxima curvatura')
print(f'{mc_menor_pendiente:.2f}, Mc menor pendiente')

In [ ]:
mc_estabilidad

## Calculo b-value

b-value estimado por Londoño (2019)

In [ ]:
m_promedio = df_filtrado_SGC1['Magnitud'].mean()
mc_londono = 1.0
b_londono = (1/(m_promedio - mc_londono)) * np.log10(np.e)

b_londono

Londoño et al. (2019) reportaron un valor de b = 1.02 y +/-0.2, con un Mc = 1.0. A continuación, se evaluarán las ecuaciones propuestas para determinar cuál de ellas reproduce un valor cercano a este estimado.

In [ ]:
N_lgr = sup.stats.obtener_N_LGR(magnitudes_array= df_filtrado_SGC1['Magnitud'],
                                magnitud_minima=0,
                                magnitud_maxima=10,
                                steps_=0.1,
                                texto=True)

N_lgr

In [ ]:
b_a_value = sup.stats.get_GutenbergRichter_values(N_count= N_lgr['N'],
                                                  magnitudes= N_lgr['m'],
                                                  mc=1.0,
                                                  magnitud_maxima=10)

b_a_value

In [ ]:
b_a_value_boost = sup.stats.get_GutenbergRichter_values_boosted(magnitudes=df_filtrado_SGC1['Magnitud'],
                                                                percentage_data= 1.0, # 0.8
                                                                magnitud_minima= 0,
                                                                magnitud_maxima= 10,
                                                                steps_= 0.1,
                                                                n_resamples= 1000,
                                                                replace_=True,
                                                                ventana_estabilidad= 10)

b_a_value_boost

In [ ]:
b_a_noise = sup.stats.get_GutenbergRichter_values_noised(N_count= N_lgr['N'],
                                                         magnitudes= N_lgr['m'],
                                                         noise=0.1,
                                                         n_resamples= 1000,
                                                         mc=1.0,
                                                         magnitud_maxima=10)

b_a_noise

In [ ]:
b_a_value_boost_2 = sup.stats.get_GutenbergRichter_values_boosted_2(magnitudes=df_filtrado_SGC1['Magnitud'],
                                                                percentage_data= 1.0, # 0.8
                                                                magnitud_minima= 0,
                                                                magnitud_maxima= 10,
                                                                steps_= 0.1,
                                                                n_resamples= 1000,
                                                                replace_=True,
                                                                correcion_m = 0.0)# no se hace la correcion de momento

b_a_value_boost_2

In [ ]:
print('b-value, a-value')
print('1.02, ---, Londoño et al. (2019)')
print(f'{b_a_value[0]:.2f}, {b_a_value[1]:.2f}, Gutenberg Richter values')
print(f'{b_a_value_boost['b_mean']:.2f}, {b_a_value_boost['a_mean']:.2f}, Gutenberg Richter boosted')
print(f'{b_a_value_boost_2['b_mean']:.2f}, {b_a_value_boost_2['a_mean']:.2f}, Gutenberg Richter boosted 2')
print(f'{b_a_noise['b_mean']:.2f}, {b_a_noise['a_mean']:.2f}, Gutenberg Richter noised')

Al comparar los resultados obtenidos con los reportados por Londoño et al. (2019), no fue posible reproducir de manera fiel sus estimaciones utilizando los valores de Mc y b-value indicados en dicho trabajo.

No obstante, al evaluar los distintos métodos probados, los valores obtenidos mediante las funciones de Gutenberg–Richter son consistentes y comparables con los estimados por los demás enfoques.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 4))

sup.plots.GutenbergRichter_scatter_plot(N_count = N_lgr['N'],
                                        magnitudes= N_lgr['m'],
                                        ax_h= ax,
                                        mc=1.0, # Londoño et al. (2919)
                                        magnitud_maxima=10)

sup.plots.GutenbergRichter_line_plot(b_value=1.16,
                                     a_value=5.73,
                                     ax_h=ax,
                                     magnitud_minima=1.0,
                                     magnitud_maxima=6)

plt.show()

# SGC-1 v2

In [ ]:
years_ = [2014, 2017] #
longitud_ = [-74.5, -73] # N
latitud_ = [6.4, 9.2] # W

In [ ]:
df_filtrado = df_unido.copy()

df_filtrado = df_filtrado[(df_filtrado['Tiempo'].dt.year >= years_[0]) & (df_filtrado['Tiempo'].dt.year <= years_[1])]
df_filtrado = df_filtrado[(df_filtrado['Longitud'] >= longitud_[0]) & (df_filtrado['Longitud'] <= longitud_[1])]
df_filtrado = df_filtrado[(df_filtrado['Latitud'] >= latitud_[0]) & (df_filtrado['Latitud'] <= latitud_[1])]

df_filtrado.describe().round(2)

In [ ]:
df_filtrado_SGC1 = df_filtrado[df_filtrado['Catalogo'] == 'SGC 1'].copy()
df_filtrado_SGC1.reset_index(drop=True, inplace=True)

df_filtrado_SGC1.describe().round(2)

In [ ]:
df_filtrado_SGC1.duplicated(subset=['Fecha-Hora UTC', 'Latitud', 'Longitud'], keep='first').sum()
# 1294 duplicados

In [ ]:
df_filtrado_SGC1['Tipo Magnitud'].unique()

In [ ]:
df_filtrado_SGC1 = df_filtrado_SGC1[df_filtrado_SGC1['Tipo Magnitud'] == 'Ml']
df_filtrado_SGC1.reset_index(drop=True, inplace=True)

df_filtrado_SGC1['Tipo Magnitud'].unique()

In [ ]:
df_filtrado_SGC1.duplicated(subset=['Fecha-Hora UTC', 'Latitud', 'Longitud'], keep='first').sum()
# Ya no se tienen duplicados

In [ ]:
df_filtrado_SGC1.describe().round(2)

## Calculo Mc

In [ ]:
N_lgr = sup.stats.obtener_N_LGR(magnitudes_array= df_filtrado_SGC1['Magnitud'],
                                magnitud_minima=0,
                                magnitud_maxima=10,
                                steps_=0.1,
                                texto=True)

N_lgr

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 4))

sup.plots.GutenbergRichter_scatter_plot(N_count = N_lgr['N'],
                                        magnitudes= N_lgr['m'],
                                        ax_h= ax,
                                        mc=1.0,
                                        magnitud_maxima=10)

sup.plots.GutenbergRichter_line_plot(b_value=1.02,
                                     a_value=6.0,
                                     ax_h=ax,
                                     magnitud_minima=1.0,
                                     magnitud_maxima=7.0)

plt.show()

In [ ]:
mc_estabilidad = sup.stats.mc_estabilidad_b(magnitudes= N_lgr['m'],
                                            valores_N = N_lgr['N'],
                                            magnitud_minima = 0,
                                            magnitud_maxima = 10,
                                            ventana_estabilidad = 10,
                                            redondeo_ = 3,
                                            min_mc_top3 = True,
                                            texto = True)


mc_estabilidad

In [ ]:
mc_max_log10 = sup.stats.mc_maximo_N_log10(magnitudes= N_lgr['m'],
                                           valores_N = N_lgr['N'],
                                           correcion_m = 0.2,
                                           magnitud_minima = 0,
                                           magnitud_maxima = 10,
                                           redondeo_N = 1)

mc_max_log10

In [ ]:
mc_max_curv = sup.stats.mc_maxima_curvatura(magnitudes_array= df_filtrado_SGC1['Magnitud'],
                                             steps_=0.1,
                                             correcion_m = 0.2,
                                             magnitud_minima = 0,
                                             magnitud_maxima = 10,
                                             texto = True)

mc_max_curv

In [ ]:
mc_menor_pendiente = sup.stats.mc_menor_pendiente(magnitudes= N_lgr['m'],
                                                  valores_N = N_lgr['N'],
                                                  magnitud_minima = 0,
                                                  magnitud_maxima = 10,
                                                  texto = True)

mc_menor_pendiente

In [ ]:
print('Valores de Mc')
print('1.00, Londoño et al. (2019)')
print(f'{mc_estabilidad['mc']:.2f}, estabilidad de b-value')
print(f'{mc_max_log10:.2f}, Mc maximo N Log10')
print(f'{mc_max_curv:.2f}, maxima curvatura')
print(f'{mc_menor_pendiente:.2f}, Mc menor pendiente')

In [ ]:
mc_estabilidad

## Calculo b-value

b-value estimado por Londoño (2019)

In [ ]:
m_promedio = df_filtrado_SGC1['Magnitud'].mean()
mc_londono = 1.0
b_londono = (1/(m_promedio - mc_londono)) * np.log10(np.e)

b_londono

In [ ]:
N_lgr = sup.stats.obtener_N_LGR(magnitudes_array= df_filtrado_SGC1['Magnitud'],
                                magnitud_minima=0,
                                magnitud_maxima=10,
                                steps_=0.1,
                                texto=True)

N_lgr

In [ ]:
b_a_value = sup.stats.get_GutenbergRichter_values(N_count= N_lgr['N'],
                                                  magnitudes= N_lgr['m'],
                                                  mc=1.0,
                                                  magnitud_maxima=10)

b_a_value

In [ ]:
b_a_value_boost = sup.stats.get_GutenbergRichter_values_boosted(magnitudes=df_filtrado_SGC1['Magnitud'],
                                                                percentage_data= 1.0, # 0.8
                                                                magnitud_minima= 0,
                                                                magnitud_maxima= 10,
                                                                steps_= 0.1,
                                                                n_resamples= 1000,
                                                                replace_=True,
                                                                ventana_estabilidad= 10)

b_a_value_boost

In [ ]:
b_a_noise = sup.stats.get_GutenbergRichter_values_noised(N_count= N_lgr['N'],
                                                         magnitudes= N_lgr['m'],
                                                         noise=0.1,
                                                         n_resamples= 1000,
                                                         mc=1.0,
                                                         magnitud_maxima=10)

b_a_noise

In [ ]:
b_a_value_boost_2 = sup.stats.get_GutenbergRichter_values_boosted_2(magnitudes=df_filtrado_SGC1['Magnitud'],
                                                                percentage_data= 1.0, # 0.8
                                                                magnitud_minima= 0,
                                                                magnitud_maxima= 10,
                                                                steps_= 0.1,
                                                                n_resamples= 1000,
                                                                replace_=True,
                                                                correcion_m = 0.0)# no se hace la correcion de momento

b_a_value_boost_2

In [ ]:
print('b-value, a-value')
print('1.02, ---, Londoño et al. (2019)')
print(f'{b_a_value[0]:.2f}, {b_a_value[1]:.2f}, Gutenberg Richter values')
print(f'{b_a_value_boost['b_mean']:.2f}, {b_a_value_boost['a_mean']:.2f}, Gutenberg Richter boosted')
print(f'{b_a_value_boost_2['b_mean']:.2f}, {b_a_value_boost_2['a_mean']:.2f}, Gutenberg Richter boosted 2')
print(f'{b_a_noise['b_mean']:.2f}, {b_a_noise['a_mean']:.2f}, Gutenberg Richter noised')

In [ ]:
b_a_value_boost['mc_mean']

In [ ]:
b_a_value_boost_2['mc_mean']

Al utilizar el catálogo completo, sin filtrar por la calidad de los eventos, el método boosted permite estimar un b-value cercano a 1.0, en concordancia con el valor reportado por Londoño et al. (2019). No obstante, la magnitud de completitud (Mc) presenta diferencias respecto a dicho estudio, con valores ligeramente mayores en nuestras pruebas (1.79 y 1.50).

En consecuencia, puede considerarse que el enfoque boosted reproduce de manera adecuada el b-value de Londoño et al. (2019), aunque no replica de forma exacta la estimación de Mc. Además, esta consistencia se alcanza únicamente al incluir todos los eventos del catálogo y no al aplicar el filtrado por calidad utilizado en Londoño et al. (2019).

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 4))

sup.plots.GutenbergRichter_scatter_plot(N_count = N_lgr['N'],
                                        magnitudes= N_lgr['m'],
                                        ax_h= ax,
                                        mc=1.7,
                                        magnitud_maxima=10)

sup.plots.GutenbergRichter_line_plot(b_value=1.0,
                                     a_value=5.9,
                                     ax_h=ax,
                                     magnitud_minima=1.0,
                                     magnitud_maxima=7.0)

plt.show()

In [ ]:
b_a_value_boostv2 = sup.stats.get_GutenbergRichter_values_boosted(magnitudes=df_filtrado_SGC1['Magnitud'],
                                                                percentage_data= 0.8, # 0.8
                                                                magnitud_minima= 0,
                                                                magnitud_maxima= 10,
                                                                steps_= 0.1,
                                                                n_resamples= 1000,
                                                                replace_=False,
                                                                ventana_estabilidad= 10)

b_a_value_boostv2

In [ ]:
b_a_value_boost_2_v2 = sup.stats.get_GutenbergRichter_values_boosted_2(magnitudes=df_filtrado_SGC1['Magnitud'],
                                                                percentage_data= 0.8, # 0.8
                                                                magnitud_minima= 0,
                                                                magnitud_maxima= 10,
                                                                steps_= 0.1,
                                                                n_resamples= 1000,
                                                                replace_=False,
                                                                correcion_m = 0.0)# no se hace la correcion de momento

b_a_value_boost_2_v2

In [ ]:
print('b-value, a-value')
print('1.02, ---, Londoño et al. (2019)')
print(f'{b_a_value[0]:.2f}, {b_a_value[1]:.2f}, Gutenberg Richter values')
print(f'{b_a_value_boost['b_mean']:.2f}, {b_a_value_boost['a_mean']:.2f}, Gutenberg Richter boosted')
print(f'{b_a_value_boostv2['b_mean']:.2f}, {b_a_value_boostv2['a_mean']:.2f}, Gutenberg Richter boosted - v2')
print(f'{b_a_value_boost_2['b_mean']:.2f}, {b_a_value_boost_2['a_mean']:.2f}, Gutenberg Richter boosted 2')
print(f'{b_a_value_boost_2_v2['b_mean']:.2f}, {b_a_value_boost_2_v2['a_mean']:.2f}, Gutenberg Richter boosted 2 - v2')
print(f'{b_a_noise['b_mean']:.2f}, {b_a_noise['a_mean']:.2f}, Gutenberg Richter noised')

In [ ]:
b_a_value_boostv2['mc_mean']

In [ ]:
b_a_value_boost_2_v2['mc_mean']

### Comparacion Boosted

Al aplicar el método boosted y, en esta ocasión, trabajar con una submuestra aleatoria del 80% del catálogo (sin reemplazo), también se obtiene un valor de b = 1.02, junto con una Mc = 1.9 y 1.5, ligeramente mayor que la estimada en la versión previa.

Esta prueba se plantea porque Londoño et al. (2019) descartaron una fracción importante de eventos: el catálogo original cuenta con 29917 registros y, tras aplicar sus filtros, se reduce a 13732, es decir, menos del 50% del total. En consecuencia, se evaluará el desempeño del enfoque boosted utilizando fracciones del catálogo entre 40% y 100%, con el fin de comparar la estabilidad de b y Mc en función del tamaño de la muestra.

In [ ]:
(13732*100) / 29917 # Londoño et al. (2019) usa solo un 43%
# de los datos disponibles

#### Con reemplazo Boosted 1

In [ ]:
boosted_40 = sup.stats.get_GutenbergRichter_values_boosted(magnitudes=df_filtrado_SGC1['Magnitud'],
                                                           percentage_data= 0.4, # 0.8
                                                           magnitud_minima= 0,
                                                           magnitud_maxima= 10,
                                                           steps_= 0.1,
                                                           n_resamples= 1000,
                                                           replace_=True,
                                                           ventana_estabilidad= 10)

boosted_40

In [ ]:
boosted_50 = sup.stats.get_GutenbergRichter_values_boosted(magnitudes=df_filtrado_SGC1['Magnitud'],
                                                           percentage_data= 0.5, # 0.8
                                                           magnitud_minima= 0,
                                                           magnitud_maxima= 10,
                                                           steps_= 0.1,
                                                           n_resamples= 1000,
                                                           replace_=True,
                                                           ventana_estabilidad= 10)

boosted_50

In [ ]:
boosted_60 = sup.stats.get_GutenbergRichter_values_boosted(magnitudes=df_filtrado_SGC1['Magnitud'],
                                                           percentage_data= 0.6, # 0.8
                                                           magnitud_minima= 0,
                                                           magnitud_maxima= 10,
                                                           steps_= 0.1,
                                                           n_resamples= 1000,
                                                           replace_=True,
                                                           ventana_estabilidad= 10)

boosted_60

In [ ]:
boosted_70 = sup.stats.get_GutenbergRichter_values_boosted(magnitudes=df_filtrado_SGC1['Magnitud'],
                                                           percentage_data= 0.7, # 0.8
                                                           magnitud_minima= 0,
                                                           magnitud_maxima= 10,
                                                           steps_= 0.1,
                                                           n_resamples= 1000,
                                                           replace_=True,
                                                           ventana_estabilidad= 10)

boosted_70

In [ ]:
boosted_80 = sup.stats.get_GutenbergRichter_values_boosted(magnitudes=df_filtrado_SGC1['Magnitud'],
                                                           percentage_data= 0.8, # 0.8
                                                           magnitud_minima= 0,
                                                           magnitud_maxima= 10,
                                                           steps_= 0.1,
                                                           n_resamples= 1000,
                                                           replace_=True,
                                                           ventana_estabilidad= 10)

boosted_80

In [ ]:
boosted_90 = sup.stats.get_GutenbergRichter_values_boosted(magnitudes=df_filtrado_SGC1['Magnitud'],
                                                           percentage_data= 0.9, # 0.8
                                                           magnitud_minima= 0,
                                                           magnitud_maxima= 10,
                                                           steps_= 0.1,
                                                           n_resamples= 1000,
                                                           replace_=True,
                                                           ventana_estabilidad= 10)

boosted_90

In [ ]:
boosted_100 = sup.stats.get_GutenbergRichter_values_boosted(magnitudes=df_filtrado_SGC1['Magnitud'],
                                                           percentage_data= 1.0, # 0.8
                                                           magnitud_minima= 0,
                                                           magnitud_maxima= 10,
                                                           steps_= 0.1,
                                                           n_resamples= 1000,
                                                           replace_=True,
                                                           ventana_estabilidad= 10)

boosted_100

In [ ]:
print('Con reemplazo')
print('Mc, b-value, a-value')
print('1.00, 1.02, ---, Londoño et al. (2019)')
print(f'{boosted_40['mc_mean']:.2f}, {boosted_40['b_mean']:.2f}, {boosted_40['a_mean']:.2f}, Boosted 40%')
print(f'{boosted_50['mc_mean']:.2f}, {boosted_50['b_mean']:.2f}, {boosted_50['a_mean']:.2f}, Boosted 50%')
print(f'{boosted_60['mc_mean']:.2f}, {boosted_60['b_mean']:.2f}, {boosted_60['a_mean']:.2f}, Boosted 60%')
print(f'{boosted_70['mc_mean']:.2f}, {boosted_70['b_mean']:.2f}, {boosted_70['a_mean']:.2f}, Boosted 70%')
print(f'{boosted_80['mc_mean']:.2f}, {boosted_80['b_mean']:.2f}, {boosted_80['a_mean']:.2f}, Boosted 80%')
print(f'{boosted_90['mc_mean']:.2f}, {boosted_90['b_mean']:.2f}, {boosted_90['a_mean']:.2f}, Boosted 90%')
print(f'{boosted_100['mc_mean']:.2f}, {boosted_100['b_mean']:.2f}, {boosted_40['a_mean']:.2f}, Boosted 100%')

In [ ]:
print('Con reemplazo')
print('Mc, b-value, a-value ... std')
print(f'{boosted_40['mc_std']:.2f}, {boosted_40['b_std']:.2f}, {boosted_40['a_std']:.2f}, Boosted 40%')
print(f'{boosted_50['mc_std']:.2f}, {boosted_50['b_std']:.2f}, {boosted_50['a_std']:.2f}, Boosted 50%')
print(f'{boosted_60['mc_std']:.2f}, {boosted_60['b_std']:.2f}, {boosted_60['a_std']:.2f}, Boosted 60%')
print(f'{boosted_70['mc_std']:.2f}, {boosted_70['b_std']:.2f}, {boosted_70['a_std']:.2f}, Boosted 70%')
print(f'{boosted_80['mc_std']:.2f}, {boosted_80['b_std']:.2f}, {boosted_80['a_std']:.2f}, Boosted 80%')
print(f'{boosted_90['mc_std']:.2f}, {boosted_90['b_std']:.2f}, {boosted_90['a_std']:.2f}, Boosted 90%')
print(f'{boosted_100['mc_std']:.2f}, {boosted_100['b_std']:.2f}, {boosted_40['a_std']:.2f}, Boosted 100%')

#### Sin reemplazo  Boosted 1

In [ ]:
boosted_40 = sup.stats.get_GutenbergRichter_values_boosted(magnitudes=df_filtrado_SGC1['Magnitud'],
                                                           percentage_data= 0.4, # 0.8
                                                           magnitud_minima= 0,
                                                           magnitud_maxima= 10,
                                                           steps_= 0.1,
                                                           n_resamples= 1000,
                                                           replace_=False,
                                                           ventana_estabilidad= 10)

boosted_40

In [ ]:
boosted_50 = sup.stats.get_GutenbergRichter_values_boosted(magnitudes=df_filtrado_SGC1['Magnitud'],
                                                           percentage_data= 0.5, # 0.8
                                                           magnitud_minima= 0,
                                                           magnitud_maxima= 10,
                                                           steps_= 0.1,
                                                           n_resamples= 1000,
                                                           replace_=False,
                                                           ventana_estabilidad= 10)

boosted_50

In [ ]:
boosted_60 = sup.stats.get_GutenbergRichter_values_boosted(magnitudes=df_filtrado_SGC1['Magnitud'],
                                                           percentage_data= 0.6, # 0.8
                                                           magnitud_minima= 0,
                                                           magnitud_maxima= 10,
                                                           steps_= 0.1,
                                                           n_resamples= 1000,
                                                           replace_=False,
                                                           ventana_estabilidad= 10)

boosted_60

In [ ]:
boosted_70 = sup.stats.get_GutenbergRichter_values_boosted(magnitudes=df_filtrado_SGC1['Magnitud'],
                                                           percentage_data= 0.7, # 0.8
                                                           magnitud_minima= 0,
                                                           magnitud_maxima= 10,
                                                           steps_= 0.1,
                                                           n_resamples= 1000,
                                                           replace_=False,
                                                           ventana_estabilidad= 10)

boosted_70

In [ ]:
boosted_80 = sup.stats.get_GutenbergRichter_values_boosted(magnitudes=df_filtrado_SGC1['Magnitud'],
                                                           percentage_data= 0.8, # 0.8
                                                           magnitud_minima= 0,
                                                           magnitud_maxima= 10,
                                                           steps_= 0.1,
                                                           n_resamples= 1000,
                                                           replace_=False,
                                                           ventana_estabilidad= 10)

boosted_80

In [ ]:
boosted_90 = sup.stats.get_GutenbergRichter_values_boosted(magnitudes=df_filtrado_SGC1['Magnitud'],
                                                           percentage_data= 0.9, # 0.8
                                                           magnitud_minima= 0,
                                                           magnitud_maxima= 10,
                                                           steps_= 0.1,
                                                           n_resamples= 1000,
                                                           replace_=False,
                                                           ventana_estabilidad= 10)

boosted_90

In [ ]:
boosted_100 = sup.stats.get_GutenbergRichter_values_boosted(magnitudes=df_filtrado_SGC1['Magnitud'],
                                                           percentage_data= 1.0, # 0.8
                                                           magnitud_minima= 0,
                                                           magnitud_maxima= 10,
                                                           steps_= 0.1,
                                                           n_resamples= 1000,
                                                           replace_=False,
                                                           ventana_estabilidad= 10)

boosted_100

In [ ]:
print('Sin reemplazo')
print('Mc, b-value, a-value')
print('1.00, 1.02, ---, Londoño et al. (2019)')
print(f'{boosted_40['mc_mean']:.2f}, {boosted_40['b_mean']:.2f}, {boosted_40['a_mean']:.2f}, Boosted 40%')
print(f'{boosted_50['mc_mean']:.2f}, {boosted_50['b_mean']:.2f}, {boosted_50['a_mean']:.2f}, Boosted 50%')
print(f'{boosted_60['mc_mean']:.2f}, {boosted_60['b_mean']:.2f}, {boosted_60['a_mean']:.2f}, Boosted 60%')
print(f'{boosted_70['mc_mean']:.2f}, {boosted_70['b_mean']:.2f}, {boosted_70['a_mean']:.2f}, Boosted 70%')
print(f'{boosted_80['mc_mean']:.2f}, {boosted_80['b_mean']:.2f}, {boosted_80['a_mean']:.2f}, Boosted 80%')
print(f'{boosted_90['mc_mean']:.2f}, {boosted_90['b_mean']:.2f}, {boosted_90['a_mean']:.2f}, Boosted 90%')
print(f'{boosted_100['mc_mean']:.2f}, {boosted_100['b_mean']:.2f}, {boosted_40['a_mean']:.2f}, Boosted 100%')

In [ ]:
print('Sin reemplazo')
print('Mc, b-value, a-value ... std')
print(f'{boosted_40['mc_std']:.2f}, {boosted_40['b_std']:.2f}, {boosted_40['a_std']:.2f}, Boosted 40%')
print(f'{boosted_50['mc_std']:.2f}, {boosted_50['b_std']:.2f}, {boosted_50['a_std']:.2f}, Boosted 50%')
print(f'{boosted_60['mc_std']:.2f}, {boosted_60['b_std']:.2f}, {boosted_60['a_std']:.2f}, Boosted 60%')
print(f'{boosted_70['mc_std']:.2f}, {boosted_70['b_std']:.2f}, {boosted_70['a_std']:.2f}, Boosted 70%')
print(f'{boosted_80['mc_std']:.2f}, {boosted_80['b_std']:.2f}, {boosted_80['a_std']:.2f}, Boosted 80%')
print(f'{boosted_90['mc_std']:.2f}, {boosted_90['b_std']:.2f}, {boosted_90['a_std']:.2f}, Boosted 90%')
print(f'{boosted_100['mc_std']:.2f}, {boosted_100['b_std']:.2f}, {boosted_40['a_std']:.2f}, Boosted 100%')

Los resultados obtenidos con y sin reemplazo son en general similares. No obstante, se observa que las desviaciones estándar estimadas para las variables Mc, b y a son ligeramente mayores cuando se emplea el método con reemplazo, lo que indica una mayor variabilidad asociada a este esquema de muestreo. Adicionalmente, en el caso del muestreo sin reemplazo, la Mc tiende a incrementarse conforme aumenta el tamaño de la muestra, lo que indica una sensibilidad de este parámetro al volumen de datos considerados.

In [ ]:
m_promedio = df_filtrado_SGC1['Magnitud'].mean()
mc_londono = 1.0
b_londono = (1/(m_promedio - mc_londono)) * np.log10(np.e)

b_londono

In [ ]:
np.log10(np.e)

#### Con reemplazo Boosted 2

In [ ]:
boosted_40 = sup.stats.get_GutenbergRichter_values_boosted_2(magnitudes=df_filtrado_SGC1['Magnitud'],
                                                                percentage_data= 0.4, # 0.8
                                                                magnitud_minima= 0,
                                                                magnitud_maxima= 10,
                                                                steps_= 0.1,
                                                                n_resamples= 1000,
                                                                replace_=True,
                                                                correcion_m = 0.0)

boosted_40

In [ ]:
boosted_50 = sup.stats.get_GutenbergRichter_values_boosted_2(magnitudes=df_filtrado_SGC1['Magnitud'],
                                                                percentage_data= 0.5, # 0.8
                                                                magnitud_minima= 0,
                                                                magnitud_maxima= 10,
                                                                steps_= 0.1,
                                                                n_resamples= 1000,
                                                                replace_=True,
                                                                correcion_m = 0.0)

boosted_50

In [ ]:
boosted_60 = sup.stats.get_GutenbergRichter_values_boosted_2(magnitudes=df_filtrado_SGC1['Magnitud'],
                                                                percentage_data= 0.6, # 0.8
                                                                magnitud_minima= 0,
                                                                magnitud_maxima= 10,
                                                                steps_= 0.1,
                                                                n_resamples= 1000,
                                                                replace_=True,
                                                                correcion_m = 0.0)

boosted_60

In [ ]:
boosted_70 = sup.stats.get_GutenbergRichter_values_boosted_2(magnitudes=df_filtrado_SGC1['Magnitud'],
                                                                percentage_data= 0.7, # 0.8
                                                                magnitud_minima= 0,
                                                                magnitud_maxima= 10,
                                                                steps_= 0.1,
                                                                n_resamples= 1000,
                                                                replace_=True,
                                                                correcion_m = 0.0)

boosted_70

In [ ]:
boosted_80 = sup.stats.get_GutenbergRichter_values_boosted_2(magnitudes=df_filtrado_SGC1['Magnitud'],
                                                                percentage_data= 0.8, # 0.8
                                                                magnitud_minima= 0,
                                                                magnitud_maxima= 10,
                                                                steps_= 0.1,
                                                                n_resamples= 1000,
                                                                replace_=True,
                                                                correcion_m = 0.0)

boosted_80

In [ ]:
boosted_90 = sup.stats.get_GutenbergRichter_values_boosted_2(magnitudes=df_filtrado_SGC1['Magnitud'],
                                                                percentage_data= 0.9, # 0.8
                                                                magnitud_minima= 0,
                                                                magnitud_maxima= 10,
                                                                steps_= 0.1,
                                                                n_resamples= 1000,
                                                                replace_=True,
                                                                correcion_m = 0.0)

boosted_90

In [ ]:
boosted_100 = sup.stats.get_GutenbergRichter_values_boosted_2(magnitudes=df_filtrado_SGC1['Magnitud'],
                                                                percentage_data= 1.0, # 0.8
                                                                magnitud_minima= 0,
                                                                magnitud_maxima= 10,
                                                                steps_= 0.1,
                                                                n_resamples= 1000,
                                                                replace_=True,
                                                                correcion_m = 0.0)

boosted_100

In [ ]:
print('Con reemplazo')
print('Mc, b-value, a-value')
print('1.00, 1.02, ---, Londoño et al. (2019)')
print(f'{boosted_40['mc_mean']:.2f}, {boosted_40['b_mean']:.2f}, {boosted_40['a_mean']:.2f}, Boosted 40%')
print(f'{boosted_50['mc_mean']:.2f}, {boosted_50['b_mean']:.2f}, {boosted_50['a_mean']:.2f}, Boosted 50%')
print(f'{boosted_60['mc_mean']:.2f}, {boosted_60['b_mean']:.2f}, {boosted_60['a_mean']:.2f}, Boosted 60%')
print(f'{boosted_70['mc_mean']:.2f}, {boosted_70['b_mean']:.2f}, {boosted_70['a_mean']:.2f}, Boosted 70%')
print(f'{boosted_80['mc_mean']:.2f}, {boosted_80['b_mean']:.2f}, {boosted_80['a_mean']:.2f}, Boosted 80%')
print(f'{boosted_90['mc_mean']:.2f}, {boosted_90['b_mean']:.2f}, {boosted_90['a_mean']:.2f}, Boosted 90%')
print(f'{boosted_100['mc_mean']:.2f}, {boosted_100['b_mean']:.2f}, {boosted_40['a_mean']:.2f}, Boosted 100%')

In [ ]:
print('Con reemplazo')
print('Mc, b-value, a-value ... std')
print(f'{boosted_40['mc_std']:.2f}, {boosted_40['b_std']:.2f}, {boosted_40['a_std']:.2f}, Boosted 40%')
print(f'{boosted_50['mc_std']:.2f}, {boosted_50['b_std']:.2f}, {boosted_50['a_std']:.2f}, Boosted 50%')
print(f'{boosted_60['mc_std']:.2f}, {boosted_60['b_std']:.2f}, {boosted_60['a_std']:.2f}, Boosted 60%')
print(f'{boosted_70['mc_std']:.2f}, {boosted_70['b_std']:.2f}, {boosted_70['a_std']:.2f}, Boosted 70%')
print(f'{boosted_80['mc_std']:.2f}, {boosted_80['b_std']:.2f}, {boosted_80['a_std']:.2f}, Boosted 80%')
print(f'{boosted_90['mc_std']:.2f}, {boosted_90['b_std']:.2f}, {boosted_90['a_std']:.2f}, Boosted 90%')
print(f'{boosted_100['mc_std']:.2f}, {boosted_100['b_std']:.2f}, {boosted_40['a_std']:.2f}, Boosted 100%')

#### Sin reemplazo  Boosted 2

In [ ]:
boosted_40 = sup.stats.get_GutenbergRichter_values_boosted_2(magnitudes=df_filtrado_SGC1['Magnitud'],
                                                                percentage_data= 0.4, # 0.8
                                                                magnitud_minima= 0,
                                                                magnitud_maxima= 10,
                                                                steps_= 0.1,
                                                                n_resamples= 1000,
                                                                replace_= False,
                                                                correcion_m = 0.0)

boosted_40

In [ ]:
boosted_50 = sup.stats.get_GutenbergRichter_values_boosted_2(magnitudes=df_filtrado_SGC1['Magnitud'],
                                                                percentage_data= 0.5, # 0.8
                                                                magnitud_minima= 0,
                                                                magnitud_maxima= 10,
                                                                steps_= 0.1,
                                                                n_resamples= 1000,
                                                                replace_= False,
                                                                correcion_m = 0.0)

boosted_50

In [ ]:
boosted_60 = sup.stats.get_GutenbergRichter_values_boosted_2(magnitudes=df_filtrado_SGC1['Magnitud'],
                                                                percentage_data= 0.6, # 0.8
                                                                magnitud_minima= 0,
                                                                magnitud_maxima= 10,
                                                                steps_= 0.1,
                                                                n_resamples= 1000,
                                                                replace_= False,
                                                                correcion_m = 0.0)

boosted_60

In [ ]:
boosted_70 = sup.stats.get_GutenbergRichter_values_boosted_2(magnitudes=df_filtrado_SGC1['Magnitud'],
                                                                percentage_data= 0.7, # 0.8
                                                                magnitud_minima= 0,
                                                                magnitud_maxima= 10,
                                                                steps_= 0.1,
                                                                n_resamples= 1000,
                                                                replace_= False,
                                                                correcion_m = 0.0)

boosted_70

In [ ]:
boosted_80 = sup.stats.get_GutenbergRichter_values_boosted_2(magnitudes=df_filtrado_SGC1['Magnitud'],
                                                                percentage_data= 0.8, # 0.8
                                                                magnitud_minima= 0,
                                                                magnitud_maxima= 10,
                                                                steps_= 0.1,
                                                                n_resamples= 1000,
                                                                replace_= False,
                                                                correcion_m = 0.0)

boosted_80

In [ ]:
boosted_90 = sup.stats.get_GutenbergRichter_values_boosted_2(magnitudes=df_filtrado_SGC1['Magnitud'],
                                                                percentage_data= 0.9, # 0.8
                                                                magnitud_minima= 0,
                                                                magnitud_maxima= 10,
                                                                steps_= 0.1,
                                                                n_resamples= 1000,
                                                                replace_= False,
                                                                correcion_m = 0.0)

boosted_90

In [ ]:
boosted_100 = sup.stats.get_GutenbergRichter_values_boosted_2(magnitudes=df_filtrado_SGC1['Magnitud'],
                                                                percentage_data= 1.0, # 0.8
                                                                magnitud_minima= 0,
                                                                magnitud_maxima= 10,
                                                                steps_= 0.1,
                                                                n_resamples= 1000,
                                                                replace_= False,
                                                                correcion_m = 0.0)

boosted_100

In [ ]:
print('Sin reemplazo')
print('Mc, b-value, a-value')
print('1.00, 1.02, ---, Londoño et al. (2019)')
print(f'{boosted_40['mc_mean']:.2f}, {boosted_40['b_mean']:.2f}, {boosted_40['a_mean']:.2f}, Boosted 40%')
print(f'{boosted_50['mc_mean']:.2f}, {boosted_50['b_mean']:.2f}, {boosted_50['a_mean']:.2f}, Boosted 50%')
print(f'{boosted_60['mc_mean']:.2f}, {boosted_60['b_mean']:.2f}, {boosted_60['a_mean']:.2f}, Boosted 60%')
print(f'{boosted_70['mc_mean']:.2f}, {boosted_70['b_mean']:.2f}, {boosted_70['a_mean']:.2f}, Boosted 70%')
print(f'{boosted_80['mc_mean']:.2f}, {boosted_80['b_mean']:.2f}, {boosted_80['a_mean']:.2f}, Boosted 80%')
print(f'{boosted_90['mc_mean']:.2f}, {boosted_90['b_mean']:.2f}, {boosted_90['a_mean']:.2f}, Boosted 90%')
print(f'{boosted_100['mc_mean']:.2f}, {boosted_100['b_mean']:.2f}, {boosted_40['a_mean']:.2f}, Boosted 100%')

In [ ]:
print('Sin reemplazo')
print('Mc, b-value, a-value ... std')
print(f'{boosted_40['mc_std']:.2f}, {boosted_40['b_std']:.2f}, {boosted_40['a_std']:.2f}, Boosted 40%')
print(f'{boosted_50['mc_std']:.2f}, {boosted_50['b_std']:.2f}, {boosted_50['a_std']:.2f}, Boosted 50%')
print(f'{boosted_60['mc_std']:.2f}, {boosted_60['b_std']:.2f}, {boosted_60['a_std']:.2f}, Boosted 60%')
print(f'{boosted_70['mc_std']:.2f}, {boosted_70['b_std']:.2f}, {boosted_70['a_std']:.2f}, Boosted 70%')
print(f'{boosted_80['mc_std']:.2f}, {boosted_80['b_std']:.2f}, {boosted_80['a_std']:.2f}, Boosted 80%')
print(f'{boosted_90['mc_std']:.2f}, {boosted_90['b_std']:.2f}, {boosted_90['a_std']:.2f}, Boosted 90%')
print(f'{boosted_100['mc_std']:.2f}, {boosted_100['b_std']:.2f}, {boosted_40['a_std']:.2f}, Boosted 100%')

Al igual que en el método Boosted 1 (que estima Mc a partir de la estabilidad del b-value) y el método Boosted 2 (donde Mc se determina mediante el criterio de máxima curvatura), los resultados obtenidos con y sin reemplazo son, en general, similares.

Se destaca que, en Boosted 2, el valor de Mc permanece constante y no presenta variaciones entre los distintos esquemas de remuestreo. Esto se debe probablemente a que el criterio de máxima curvatura identifica un punto asociado al máximo en la distribución del número de eventos, el cual no se ve afectado por los submuestreos, ya que estos no alteran de manera significativa la proporción ni la relación relativa entre las magnitudes. Como consecuencia, la desviación estándar de Mc es igual a 0.0.

Adicionalmente, las desviaciones estándar de los parámetros estimados mediante Boosted 2 son menores que las obtenidas con Boosted 1, lo que sugiere una mayor estabilidad del primero bajo los distintos esquemas de remuestreo.

In [ ]:
m_promedio = df_filtrado_SGC1['Magnitud'].mean()
mc_londono = 1.0
b_londono = (1/(m_promedio - mc_londono)) * np.log10(np.e)

b_londono

In [ ]:
np.log10(np.e)

# 50 km

En el trabajo de Londoño (2019) se da ha entender que el calculo solo se hace para los eventos de menos de 50 km de profundidad, por lo que se va tomar todo el catalogo y estos eventos.

In [ ]:
years_ = [2014, 2017] #
longitud_ = [-74.5, -73] # N
latitud_ = [6.4, 9.2] # W
prof_max = 50 # km
rms_ = 0.4 # s
erros_loc_ = 10 # km
gap_ = 180
ml_ = [0.1, 5.7]

In [ ]:
df_filtrado = df_unido.copy()

df_filtrado = df_filtrado[(df_filtrado['Tiempo'].dt.year >= years_[0]) & (df_filtrado['Tiempo'].dt.year <= years_[1])]
df_filtrado = df_filtrado[(df_filtrado['Longitud'] >= longitud_[0]) & (df_filtrado['Longitud'] <= longitud_[1])]
df_filtrado = df_filtrado[(df_filtrado['Latitud'] >= latitud_[0]) & (df_filtrado['Latitud'] <= latitud_[1])]
df_filtrado = df_filtrado[df_filtrado['Profundidad [km]'] <= prof_max]

df_filtrado.describe().round(2)

In [ ]:
df_filtrado_SGC1 = df_filtrado[df_filtrado['Catalogo'] == 'SGC 1'].copy()
df_filtrado_SGC1.reset_index(drop=True, inplace=True)

df_filtrado_SGC1.describe().round(2)

In [ ]:
df_filtrado_SGC1.duplicated(subset=['Fecha-Hora UTC', 'Latitud', 'Longitud'], keep='first').sum()
# 1293 duplicados

In [ ]:
df_filtrado_SGC1['Tipo Magnitud'].unique()

In [ ]:
df_filtrado_SGC1 = df_filtrado_SGC1[df_filtrado_SGC1['Tipo Magnitud'] == 'Ml']
df_filtrado_SGC1.reset_index(drop=True, inplace=True)

df_filtrado_SGC1['Tipo Magnitud'].unique()

In [ ]:
df_filtrado_SGC1.duplicated(subset=['Fecha-Hora UTC', 'Latitud', 'Longitud'], keep='first').sum()
# Ya no se tienen duplicados

In [ ]:
df_filtrado_SGC1.describe().round(2)

## Calculo Mc

In [ ]:
N_lgr = sup.stats.obtener_N_LGR(magnitudes_array= df_filtrado_SGC1['Magnitud'],
                                magnitud_minima=0,
                                magnitud_maxima=10,
                                steps_=0.1,
                                texto=True)

N_lgr

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 4))

sup.plots.GutenbergRichter_scatter_plot(N_count = N_lgr['N'],
                                        magnitudes= N_lgr['m'],
                                        ax_h= ax,
                                        mc=1.0,
                                        magnitud_maxima=10)

sup.plots.GutenbergRichter_line_plot(b_value=1.02,
                                     a_value=4.3,
                                     ax_h=ax,
                                     magnitud_minima=1.0,
                                     magnitud_maxima=4.7)

plt.show()

In [ ]:
mc_estabilidad = sup.stats.mc_estabilidad_b(magnitudes= N_lgr['m'],
                                            valores_N = N_lgr['N'],
                                            magnitud_minima = 0,
                                            magnitud_maxima = 10,
                                            ventana_estabilidad = 10,
                                            redondeo_ = 3,
                                            min_mc_top3 = True,
                                            texto = True)


mc_estabilidad

In [ ]:
mc_max_log10 = sup.stats.mc_maximo_N_log10(magnitudes= N_lgr['m'],
                                           valores_N = N_lgr['N'],
                                           correcion_m = 0.2,
                                           magnitud_minima = 0,
                                           magnitud_maxima = 10,
                                           redondeo_N = 1)

mc_max_log10

In [ ]:
mc_max_curv = sup.stats.mc_maxima_curvatura(magnitudes_array= df_filtrado_SGC1['Magnitud'],
                                             steps_=0.1,
                                             correcion_m = 0.2,
                                             magnitud_minima = 0,
                                             magnitud_maxima = 10,
                                             texto = True)

mc_max_curv

In [ ]:
mc_menor_pendiente = sup.stats.mc_menor_pendiente(magnitudes= N_lgr['m'],
                                                  valores_N = N_lgr['N'],
                                                  magnitud_minima = 0,
                                                  magnitud_maxima = 10,
                                                  texto = True)

mc_menor_pendiente

In [ ]:
print('Valores de Mc')
print('1.00, Londoño et al. (2019)')
print(f'{mc_estabilidad['mc']:.2f}, estabilidad de Mc')
print(f'{mc_max_log10:.2f}, Mc maximo N Log10')
print(f'{mc_max_curv:.2f}, maxima curvatura')
print(f'{mc_menor_pendiente:.2f}, Mc menor pendiente')

In [ ]:
mc_estabilidad

## Calculo b-value

b-value estimado por Londoño (2019)

In [ ]:
m_promedio = df_filtrado_SGC1['Magnitud'].mean()
mc_londono = 1.0
b_londono = (1/(m_promedio - mc_londono)) * np.log10(np.e)

b_londono

In [ ]:
N_lgr = sup.stats.obtener_N_LGR(magnitudes_array= df_filtrado_SGC1['Magnitud'],
                                magnitud_minima=0,
                                magnitud_maxima=10,
                                steps_=0.1,
                                texto=True)

N_lgr

In [ ]:
b_a_value = sup.stats.get_GutenbergRichter_values(N_count= N_lgr['N'],
                                                  magnitudes= N_lgr['m'],
                                                  mc=1.0,
                                                  magnitud_maxima=10)

b_a_value

In [ ]:
b_a_value_boost = sup.stats.get_GutenbergRichter_values_boosted(magnitudes=df_filtrado_SGC1['Magnitud'],
                                                                percentage_data= 1.0, # 0.8
                                                                magnitud_minima= 0,
                                                                magnitud_maxima= 10,
                                                                steps_= 0.1,
                                                                n_resamples= 1000,
                                                                replace_=True,
                                                                ventana_estabilidad= 10)

b_a_value_boost

In [ ]:
b_a_noise = sup.stats.get_GutenbergRichter_values_noised(N_count= N_lgr['N'],
                                                         magnitudes= N_lgr['m'],
                                                         noise=0.1,
                                                         n_resamples= 1000,
                                                         mc=1.0,
                                                         magnitud_maxima=10)

b_a_noise

In [ ]:
b_a_value_boost_2 = sup.stats.get_GutenbergRichter_values_boosted_2(magnitudes=df_filtrado_SGC1['Magnitud'],
                                                                percentage_data= 1.0, # 0.8
                                                                magnitud_minima= 0,
                                                                magnitud_maxima= 10,
                                                                steps_= 0.1,
                                                                n_resamples= 1000,
                                                                replace_=True,
                                                                correcion_m = 0.0)# no se hace la correcion de momento

b_a_value_boost_2

In [ ]:
print('b-value, a-value')
print('1.02, ---, Londoño et al. (2019)')
print(f'{b_a_value[0]:.2f}, {b_a_value[1]:.2f}, Gutenberg Richter values')
print(f'{b_a_value_boost['b_mean']:.2f}, {b_a_value_boost['a_mean']:.2f}, Gutenberg Richter boosted')
print(f'{b_a_value_boost_2['b_mean']:.2f}, {b_a_value_boost_2['a_mean']:.2f}, Gutenberg Richter boosted 2')
print(f'{b_a_noise['b_mean']:.2f}, {b_a_noise['a_mean']:.2f}, Gutenberg Richter noised')

In [ ]:
b_a_value_boost['mc_mean']

In [ ]:
b_a_value_boost_2['mc_mean']

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 4))

# Boosted 2
sup.plots.GutenbergRichter_scatter_plot(N_count = N_lgr['N'],
                                        magnitudes= N_lgr['m'],
                                        ax_h= ax,
                                        mc=1.5,
                                        magnitud_maxima=10)

sup.plots.GutenbergRichter_line_plot(b_value=1.01,
                                     a_value=4.32,
                                     ax_h=ax,
                                     magnitud_minima=1.0,
                                     magnitud_maxima=4.7)

plt.show()

In [ ]:
b_a_value_boostv2 = sup.stats.get_GutenbergRichter_values_boosted(magnitudes=df_filtrado_SGC1['Magnitud'],
                                                                percentage_data= 0.8, # 0.8
                                                                magnitud_minima= 0,
                                                                magnitud_maxima= 10,
                                                                steps_= 0.1,
                                                                n_resamples= 1000,
                                                                replace_=False,
                                                                ventana_estabilidad= 10)

b_a_value_boostv2

In [ ]:
b_a_value_boost_2_v2 = sup.stats.get_GutenbergRichter_values_boosted_2(magnitudes=df_filtrado_SGC1['Magnitud'],
                                                                percentage_data= 0.8, # 0.8
                                                                magnitud_minima= 0,
                                                                magnitud_maxima= 10,
                                                                steps_= 0.1,
                                                                n_resamples= 1000,
                                                                replace_=True,
                                                                correcion_m = 0.0)# no se hace la correcion de momento

b_a_value_boost_2_v2

In [ ]:
print('b-value, a-value')
print('1.02, ---, Londoño et al. (2019)')
print(f'{b_a_value[0]:.2f}, {b_a_value[1]:.2f}, Gutenberg Richter values')
print(f'{b_a_value_boost['b_mean']:.2f}, {b_a_value_boost['a_mean']:.2f}, Gutenberg Richter boosted')
print(f'{b_a_value_boostv2['b_mean']:.2f}, {b_a_value_boostv2['a_mean']:.2f}, Gutenberg Richter boosted - v2') # 0.8
print(f'{b_a_value_boost_2['b_mean']:.2f}, {b_a_value_boost_2['a_mean']:.2f}, Gutenberg Richter boosted 2')
print(f'{b_a_value_boost_2_v2['b_mean']:.2f}, {b_a_value_boost_2_v2['a_mean']:.2f}, Gutenberg Richter boosted 2 - v2') # 0.8
print(f'{b_a_noise['b_mean']:.2f}, {b_a_noise['a_mean']:.2f}, Gutenberg Richter noised')

# Fin